Copyright 2023 DeepMind Technologies Limited

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# RoboVQA Data Loading and Eval

This colab contains example code for interacting with the RoboVQA dataset for
training and evaluation.

For more experiment results and details about the dataset, see [robovqa.github.io](https://robovqa.github.io)

To cite,
```bibtex
@inproceedings{robovqa2023arxiv,
    title={RoboVQA: Multimodal Long-Horizon Reasoning for Robotics},
    author={Pierre Sermanet and Tianli Ding and Jeffrey Zhao and Fei Xia and Debidatta Dwibedi and Keerthana Gopalakrishnan and Christine Chan and Gabriel Dulac-Arnold and Sharath Maddineni and Nikhil J Joshi and Pete Florence and Wei Han and Robert Baruch and Yao Lu and Suvir Mirchandani and Peng Xu and Pannag Sanketi and Karol Hausman and Izhak Shafran and Brian Ichter and Yuan Cao},
    booktitle={arXiv preprint arXiv:2311.00899},
    year={2023}
}
```

## Explore examples in the dataset

In [ ]:
# #@title Display utils - Not used for evaluation
# from IPython import display
# from PIL import Image
# import matplotlib.pyplot as plt

# def as_gif(images):
#   # Render the images as the gif:
#   images[0].save('/tmp/temp.gif', save_all=True, append_images=images[1:], duration=100, loop=0)
#   gif_bytes = open('/tmp/temp.gif','rb').read()
#   return gif_bytes


# def display_image(images):
#   if len(images) >1 :
#     result_images = []
#     for x in images:
#       result_images.append(Image.fromarray(x))
#     display.display(display.Image(as_gif(result_images)))
#   else:
#     plt_image =Image.fromarray(images[0])
#     plt.imshow(plt_image)
#     plt.show()

In [ ]:
#@title Task utils
"""Tasks related utils."""

import random
import re
from absl import logging


class Task:
  """A class for handling tags and splits in a given task."""

  # Tags for default splitting, based on who is talking.
  PRED_STARTS = ['Robot:', 'Thought:', 'Action:']
  NOPRED_STARTS = ['User:', 'System:']

  # Tags surrounding all blocks needing to be predicted by the model.
  PRED_START = '<PRED>'
  PRED_END = '</PRED>'
  # Tags surrounding only binary answers, typically 'yes' and 'no'.
  PRED_ANSWER_BINARY_START = '<PRED:ANSWER:BINARY>'
  PRED_ANSWER_BINARY_END = '</PRED:ANSWER:BINARY>'
  # Tags surrounding all discrete answers coming from a limited set of classes,
  # e.g. 'yes', 'no', 'halfway there', 'done', '10s', etc.
  PRED_ANSWER_DISCRETE_START = '<PRED:ANSWER:DISCRETE>'
  PRED_ANSWER_DISCRETE_END = '</PRED:ANSWER:DISCRETE>'
  # Tags surrounding things that constitute an answer to a question,
  # the question may be asked by a user or by the model itself.
  PRED_ANSWER_START = '<PRED:ANSWER'
  PRED_ANSWER_END = '</PRED:ANSWER'
  # Tags that have any sort of short-content value
  PRED_ALL_START = '<PRED:'
  PRED_ALL_END = '</PRED:'

  TAGS_RE = r'(</*\w[:\w]*>)'

  def __init__(self, text):
    self.text = text

  def get_random_split(self, split_type='speaker'):
    splits = self.get_splits(split_type)
    return random.choice(splits)

  def get_splits(self, split_type='speaker'):
    """Returns a list of (source, target) split pairs."""
    if split_type == 'pred':
      return self.get_splits_from_tags(
          start_tags=[self.PRED_START], end_tags=[self.PRED_END])
    elif split_type == 'binary':
      return self.get_splits_from_tags(
          start_tags=[self.PRED_BINARY_START], end_tags=[self.PRED_BINARY_END])
    elif split_type == 'discrete':
      return self.get_splits_from_tags(
          start_tags=[self.PRED_DISCRETE_START],
          end_tags=[self.PRED_DISCRETE_END])
    elif split_type == 'answer':
      return self.get_splits_from_tags(
          start_tags=[self.PRED_ANSWER_START], end_tags=[self.PRED_ANSWER_END])
    elif split_type == 'A:':
      return self.get_splits_from_tags(start_tags=['A:'], end_tags=[])
    elif split_type == 'speaker':
      return self.get_splits_from_tags(
          start_tags=self.PRED_STARTS, end_tags=self.NOPRED_STARTS)
    elif split_type == 'all':
      return self.get_splits_from_tags(
          start_tags=[self.PRED_ALL_START], end_tags=[self.PRED_ALL_END]
      )
    else:
      raise ValueError('Unknown split type: %s' % split_type)

  def get_splits_from_tags(self, start_tags, end_tags):
    """Returns a list of (source, target) split pairs given start/end tags."""
    # Find all the first positions of a start element.
    split_positions = []
    position = 0
    while position < len(self.text):
      # Find the next start tag given current position.
      start_position = self.find_next_tag(position, start_tags)
      if start_position is None:
        break
      # Then find the first end tag after this start tag.
      end_position = self.find_next_tag(start_position, end_tags)
      if end_position is None:
        end_position = len(self.text)
      split_positions.append((start_position, end_position))
      position = end_position + 1
    return self.get_splits_from_positions(split_positions)

  def get_splits_from_positions(self, split_positions):
    """Returns a list of (source, target) split pairs given split positions."""
    # Create splits.
    splits = []
    for (split_position, end_position) in split_positions:
      source = ''
      if split_position > 0:
        source = self.text[:split_position]
        source = self._remove_tags(source)
      target = self.text[split_position:end_position]
      target = self._remove_tags(target)
      splits.append((source, target))

    # If no splits are found, return entire text.
    if not splits:
      splits = [('', self.text)]

    return splits

  def find_next_tag(self, position, tags):
    tag_position = None
    lower_text = self.text.lower()
    for tag in tags:
      p = lower_text.find(tag.lower(), position)
      if p >= 0 and (tag_position is None or p < tag_position):
        tag_position = p
    return tag_position

  def _remove_tags(self, text):
    return re.sub(self.TAGS_RE, '', text)

  def remove_tags(self):
    self.text = self._remove_tags(self.text)

  def __str__(self):
    return self.text


class Tasks():
  """A class for handling and holding tasks information."""

  TASK_RE = r'(<task[:\w]*>)'
  RE_FLAGS = re.IGNORECASE

  def __init__(self, tasks_raw=None):
    # Contains all tasks for each task type in this Tasks collection
    # key: str, task type (tag)
    # value: list[str], question-answers which belong to this task type
    self.tasks_dict = {}
    self.tasks_list = []
    self.tasks_types = []
    self.tasks_raw = tasks_raw
    if tasks_raw is not None:
      self.add(tasks_raw)

  def add(self, tasks):
    self.add_from_text(tasks)

  def add_from_dict(self, tasks_dict):
    for name, tasks in tasks_dict.items():
      if name not in self.tasks_dict:
        self.tasks_dict[name] = []
      self.tasks_dict[name].extend(tasks)
      self.tasks_list.extend(tasks)
      self.tasks_types.extend([name] * len(tasks))

  def add_from_text(self, text):
    task_dict = self.text_to_dict(text)
    self.add_from_dict(task_dict)

  def text_to_dict(self, text):
    """Returns all tasks associated with this video."""
    # Split a serialized string into raw strings of individual tasks
    split = re.split(self.TASK_RE, text, flags=self.RE_FLAGS)[1:]
    # Construct a dict of
    # key: str, task type (tag)
    # value: list[str], question-answers which belong to this task type
    tasks_dict = {}
    i = 0
    while i < len(split) - 1:
      tag = split[i].strip()
      task = split[i+1].lstrip()
      if task:
        if tag not in tasks_dict:
          tasks_dict[tag] = []
        tasks_dict[tag].append(task)
      i += 2
    return tasks_dict

  def __str__(self, show_tasks=True):
    s_parts = []
    s_parts.append('%d task types in %d tasks:\n' % (
        len(self.tasks_dict.keys()), len(self)))
    for key in sorted(self.tasks_dict):
      tasks = self.tasks_dict[key]
      s_parts.append('%s (%d / %d, %.1f%%)' % (
          key, len(tasks), len(self), 100 * len(tasks) / float(len(self))))
      if show_tasks:
        s_parts.append('\n\t%s' % str(tasks))
      s_parts.append('\n')
    return ''.join(s_parts)

  def detailed_str(self):
    s_parts = []
    s_parts.append('Raw input: %s' % str(self.tasks_raw))
    s_parts.append('\n%s' % self.__str__(show_tasks=True))
    return ''.join(s_parts)

  def get_stats(self):
    return self.__str__(show_tasks=False)

  def __len__(self):
    return len(self.tasks_list)

  def get_tasks_list(self):
    return self.tasks_list

  def get_tasks_types(self):
    return self.tasks_types

  def get_random_task(self):
    if not self.tasks_list:
      raise ValueError('Unexpected empty tasks list')
    return random.choice(self.tasks_list)

  def sample_task(self, weights):
    """Sample a task using weights associated with task patterns.

    Note: only tasks matching the patterns in the weights dictionary will
    be considered, the patterns for which no tasks are found will be ignored.

    Args:
      weights: a dict assigning weights to task patterns, e.g.
        {'<task:success:.*': .1}.
    Returns:
      Str, a task string (without task tag).
    """
    # Organize matching tasks by pattern.
    matching_tasks = {}
    matching_weights = {}
    for pattern, weight in weights.items():
      for task, tasks in self.tasks_dict.items():
        if re.fullmatch(pattern, task, flags=self.RE_FLAGS):
          if pattern not in matching_tasks:
            matching_tasks[pattern] = []
            matching_weights[pattern] = weight
          matching_tasks[pattern].extend(tasks)

    # Sample a pattern given weights.
    if not matching_weights.keys():
      logging.warning('No tasks matching weights %s in %s: %s',
                      str(weights), self.detailed_str(), str(matching_weights))
      return None, None
    pattern = random.choices(
        list(matching_weights.keys()), list(matching_weights.values()))[0]

    # Sample a task given a pattern.
    tasks = matching_tasks[pattern]
    if not tasks:
      raise ValueError('No tasks to sample of type %s in %s'
                       % (pattern, self.tasks_raw))
    task = random.choice(tasks)
    if not task:
      raise ValueError((
          'No tasks ("%s") is returned after choosing pattern %s and'
          ' returning random from "%s" from %s') % (
              task, pattern, matching_tasks[pattern], self.detailed_str()))
    return task, pattern


def fetch_question_answer(text):
  tasks = Tasks(text)
  results = []
  for i, (task_type, tasks) in enumerate(tasks.tasks_dict.items()):
    for task in tasks:
      t = Task(task)
      splits = t.get_splits('A:')
      for split in splits:
        question, answer = split
        question = question.strip()
        answer = answer.strip()
        results.append((i, task_type, question, answer))
  return results


# def display_text(text):
#   question_answers = fetch_question_answer(text)
#   for i, task_type, question, answer in question_answers:
#     print('Task %d (type: %s): %s %s' % (i, task_type, question, answer))

In [ ]:
#@title Grab the dataset from Google Cloud Storage
import tensorflow as tf

filepaths = tf.io.gfile.glob('gs://gdm-robovqa/tfrecord/val/val*')

In [ ]:
#@title  Create a TF Dataset
dataset = tf.data.TFRecordDataset(filepaths)
np_iter = dataset.as_numpy_iterator()

In [ ]:
#@title Let's look at an example in this dataset
raw_record = next(np_iter)
example = tf.train.SequenceExample()
example.ParseFromString(raw_record)

images = []
for bl in example.feature_lists.feature_list.get('images').feature:
  code = bl.bytes_list.value[0]
  image = tf.image.decode_jpeg(code).numpy()
  images.append(image)
# print('Displayed items are:')
# print('Video image frames (in GIF format)')
# print('All VQA tasks, in format of "<Task num> (task type): <Question> <Answer>"')
# display_image(images)
# display_text(example.feature_lists.feature_list.get("texts").feature[0].bytes_list.value[0].decode('utf-8'))


## Evaluate against the dataset using BLEU score

In [ ]:
#@title Additional imports

!pip install sacrebleu
import sacrebleu

In [ ]:
import sacrebleu

def mock_call_model(images, question):
  # Replace this with your model implementation!
  return 'A: place the packet on the table'


def get_eval_example():
  eval_filepaths = tf.io.gfile.glob('gs://gdm-robovqa/tfrecord/val/val*')
  eval_dataset = tf.data.TFRecordDataset(eval_filepaths)
  eval_np_iter = eval_dataset.as_numpy_iterator()
  eval_raw_record = next(eval_np_iter)
  eval_example = tf.train.SequenceExample()
  eval_example.ParseFromString(eval_raw_record)
  return eval_example


def run_eval(call_model):
  example = get_eval_example()
  images = []
  print('Below are sampled eval answers and BLEU scores')
  for bl in example.feature_lists.feature_list.get('images').feature:
    code = bl.bytes_list.value[0]
    image = tf.image.decode_jpeg(code).numpy()
    images.append(image)
  qa_list = fetch_question_answer(example.feature_lists.feature_list.get("texts").feature[0].bytes_list.value[0].decode('utf-8'))
  for _, _, question, answer in qa_list:
    pred_answer = call_model(images, question)
    bleu = sacrebleu.sentence_bleu(pred_answer, [answer])
    print(f'Question: {question}\nAnswer: {answer}\nPredicted Answer: {pred_answer}\nBLEU: {bleu}%')


In [ ]:
run_eval(mock_call_model)

In [ ]:
eval_filepaths = tf.io.gfile.glob('gs://gdm-robovqa/tfrecord/val/val*')
eval_dataset = tf.data.TFRecordDataset(eval_filepaths)

In [ ]:
def count_qa_pairs(dataset):
  total_qa_pairs = 0
  for raw_record in dataset.as_numpy_iterator():
    example = tf.train.SequenceExample()
    example.ParseFromString(raw_record)
    qa_list = fetch_question_answer(example.feature_lists.feature_list.get("texts").feature[0].bytes_list.value[0].decode('utf-8'))
    total_qa_pairs += len(qa_list)
  return total_qa_pairs

validation_qa_count = count_qa_pairs(eval_dataset)
print(f'Total number of QA pairs in the validation dataset: {validation_qa_count}')

In [ ]:
from collections import Counter

# Re-initialize the dataset iterator to start from the beginning for analysis
eval_filepaths = tf.io.gfile.glob('gs://gdm-robovqa/tfrecord/val/val*')
eval_dataset_for_analysis = tf.data.TFRecordDataset(eval_filepaths)

task_types_counter = Counter()

for raw_record in eval_dataset_for_analysis.as_numpy_iterator():
  example = tf.train.SequenceExample()
  example.ParseFromString(raw_record)
  qa_list = fetch_question_answer(example.feature_lists.feature_list.get("texts").feature[0].bytes_list.value[0].decode('utf-8'))
  for _, task_type, _, _ in qa_list:
    task_types_counter[task_type] += 1

print("Distribution of Task Types in the Validation Dataset:")
for task_type, count in task_types_counter.most_common():
  print(f"  {task_type}: {count}")

As you can see, the dataset contains various task types, which the `Task` and `Tasks` utility classes help categorize and extract. Each `task_type` represents a different kind of question or instruction given to the model.

For instance, `<task:planning:freeform>` seems to be the most common, indicating open-ended planning questions. Other types like `<task:action:pick>` or `<task:action:place>` suggest specific action-oriented queries. The system uses these tags to define what kind of information is being requested or provided.

Let's also look at an example of how the `Task` class itself extracts question and answer pairs from a raw text string, which is what `fetch_question_answer` does internally.

In [ ]:
# Get one example raw text from the dataset to demonstrate the split
one_raw_record = next(iter(eval_dataset))
one_example = tf.train.SequenceExample()
one_example.ParseFromString(one_raw_record.numpy())
raw_text = one_example.feature_lists.feature_list.get("texts").feature[0].bytes_list.value[0].decode('utf-8')

print("\n--- Raw Text Example ---")
print(raw_text)

print("\n--- Extracted QA Pairs (using fetch_question_answer) ---")
qa_pairs_example = fetch_question_answer(raw_text)
for i, task_type, question, answer in qa_pairs_example:
    print(f"Task Type: {task_type}")
    print(f"  Question: {question}")
    print(f"  Answer: {answer}")

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
import torch

In [ ]:
model_path = "shreethar/stage1_unsloth"

In [ ]:
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    dtype = torch.bfloat16,
    device_map = "cuda"
)

In [ ]:
processor = AutoProcessor.from_pretrained(model_path)

In [ ]:
frames = images

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "video": frames},
            {"type": "text", "text": question}
        ]
    }
]

In [ ]:
inputs = processor.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False
)

In [ ]:
print(inputs)

In [ ]:
inputs = processor.apply_chat_template(
    messages,
    tokenize = True,
    return_dict = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    enable_thinking = False,
).to(model.device)

In [ ]:
output = model.generate(
    **inputs,
    max_new_tokens = 16,
    stop_strings=["<|im_end|>"],
    eos_token_id=processor.tokenizer.convert_tokens_to_ids("<|im_end|>"),
    repetition_penalty=1.2,   # penalises exact repeats
    tokenizer=processor.tokenizer
    )

In [ ]:
print(processor.tokenizer.decode(output[0]))

In [ ]:
import torch # Ensure torch is imported if not already

In [ ]:
def call_model_actual(images_input, question_input):
  question = question_input
  frames = images_input

  messages = [
      {
          "role": "user",
          "content": [
              {"type": "video", "video": frames},
              {"type": "text", "text": question}
          ]
      }
  ]

  inputs = processor.apply_chat_template(
      messages,
      tokenize=True,
      return_dict=True,
      add_generation_prompt=True,
      return_tensors="pt",
      enable_thinking=False,
  )

  for k, v in inputs.items():
      if isinstance(v, torch.Tensor):
          inputs[k] = v.to(model.device)

  # Get the length of the input tokens (prompt)
  input_ids_len = inputs['input_ids'].shape[1]

  # Generate output token IDs
  output_ids = model.generate(
      **inputs,
      max_new_tokens=16,
      stop_strings=["<|im_end|>"],
      eos_token_id=processor.tokenizer.convert_tokens_to_ids("<|im_end|>"),
      repetition_penalty=1.2,
      tokenizer=processor.tokenizer # Re-added the tokenizer argument
  )

  generated_tokens = output_ids[0, input_ids_len:]

  # Decode only the generated tokens, skipping special tokens
  pred_answer_raw = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)

  # Refined cleaning logic based on user's observation:
  # "its always </think>\n\nanswer_goes_here<|im_end|>"

  final_answer = pred_answer_raw

  # First, remove everything up to and including the specific separator "</think>\n\n"
  clean_answer_parts = final_answer.split('</think>\n\n', 1) # Split only once

  if len(clean_answer_parts) > 1:
      # If the separator was found, the answer is the second part
      final_answer = clean_answer_parts[1]
  # else: if separator is not found, final_answer remains pred_answer_raw

  # Strip any leading/trailing whitespace
  final_answer = final_answer.strip()

  # Remove potential trailing <|im_end|> if skip_special_tokens didn't catch it
  if final_answer.endswith('<|im_end|>'):
      final_answer = final_answer[:-len('<|im_end|>')].strip()

  # Also, if the model includes its own prompt like 'assistant:' or 'A:', strip that
  if final_answer.lower().startswith('assistant:'):
      final_answer = final_answer[len('assistant:'):].strip()
  if final_answer.lower().startswith('a:'):
      final_answer = final_answer[len('a:'):].strip()

  return final_answer

In [ ]:
def run_eval_for_first_n_outputs(call_model_func, num_examples=5):
  eval_filepaths = tf.io.gfile.glob('gs://gdm-robovqa/tfrecord/val/val*')
  eval_dataset_limited = tf.data.TFRecordDataset(eval_filepaths).take(num_examples)
  eval_np_iter = eval_dataset_limited.as_numpy_iterator()

  print(f'Below are sampled eval answers and BLEU scores for the first {num_examples} outputs:')
  example_count = 0
  for eval_raw_record in eval_np_iter:
    eval_example = tf.train.SequenceExample()
    eval_example.ParseFromString(eval_raw_record)

    images = []
    for bl in eval_example.feature_lists.feature_list.get('images').feature:
      code = bl.bytes_list.value[0]
      image = tf.image.decode_jpeg(code).numpy()
      images.append(image)

    qa_list = fetch_question_answer(eval_example.feature_lists.feature_list.get("texts").feature[0].bytes_list.value[0].decode('utf-8'))

    for _, _, question, answer in qa_list:
      if example_count >= num_examples:
        break

      pred_answer_raw = call_model_func(images, question)
      # Extract the predicted answer content, similar to how the actual answer is 'A: ...'
      # This might need refinement based on the exact output format of your model
      # For now, let's assume the model generates text that might contain 'A: ' or similar.
      # For a more robust solution, you'd parse the model's output more carefully.
      # Simple extraction: remove everything before 'A:' if present, or take the whole string.
      if 'A:' in pred_answer_raw:
          pred_answer = pred_answer_raw.split('A:', 1)[1].strip()
      elif 'assistant' in pred_answer_raw:
          # This is a heuristic, may need adjustment based on actual model output structure
          pred_answer = pred_answer_raw.split('assistant', 1)[1].strip()
      else:
          pred_answer = pred_answer_raw.strip()

      # The reference answer often includes 'A: ' prefix, which should be removed for BLEU calculation
      # if the model's prediction does not include it as part of the 'answer'.
      reference_answer = answer.replace('A:', '').strip()

      bleu = sacrebleu.sentence_bleu(pred_answer, [reference_answer])
      print(f'\n--- Example {example_count + 1} ---')
      print(f'Question: {question}')
      print(f'Actual Answer: {answer}')
      print(f'Predicted Answer (decoded): {pred_answer_raw}')
      print(f'Predicted Answer (for BLEU): {pred_answer}')
      print(f'Reference Answer (for BLEU): {reference_answer}')
      print(f'BLEU: {bleu}')
      example_count += 1
    if example_count >= num_examples:
      break

Now, let's run the evaluation for the first 5 outputs using the actual model.

In [ ]:
run_eval_for_first_n_outputs(call_model_actual, num_examples=5)